In [33]:
import os
import asyncio
from IPython.display import display,Markdown

In [27]:
import resend
from agents import Agent,Runner,function_tool
from dotenv import load_dotenv,find_dotenv
from openai import AsyncOpenAI,OpenAI
from agents import set_default_openai_client,OpenAIChatCompletionsModel,trace,set_tracing_export_api_key

In [24]:
load_dotenv(find_dotenv(),override=True)

True

In [25]:
resend.api_key = os.getenv("RESEND_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")

In [28]:
tracing_api_key = os.getenv("OPENAI_API_KEY")
set_tracing_export_api_key(tracing_api_key)

In [29]:
client_gemini = AsyncOpenAI(base_url=gemini_base_url, api_key=gemini_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3-flash-preview",openai_client=client_gemini)

In [36]:
instructions1 = "You are a sales agent working for CoolAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails. Return Only HTML MAIL NOTHING ELSE"

instructions2 = "You are a humorous, engaging sales agent working for CoolAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response. Return Only HTML MAIL NOTHING ELSE"

instructions3 = "You are a busy sales agent working for CoolAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails. Return Only HTML MAIL NOTHING ELSE"

In [37]:
sales_agent1 = Agent(name="Professional Sales Agent",instructions=instructions1,model=gemini_model)
sales_agent2 = Agent(name="Engaging Sales Agent",instructions=instructions2,model=gemini_model)
sales_agent3 = Agent(name="Busy Sales Agent",instructions=instructions3,model=gemini_model)

In [34]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

In [35]:
for output in outputs:
    print(output + "\n\n")

Subject: Optimizing [Company Name]’s SOC2 Audit Readiness

Dear [Prospect Name],

The administrative burden of achieving and maintaining SOC2 compliance often acts as a significant drain on engineering and legal resources, diverting focus from core product development and strategic growth.

I am reaching out from CoolAI to discuss how we are leveraging artificial intelligence to transform the compliance landscape. Unlike manual, spreadsheet-heavy processes, CoolAI automates evidence collection and continuous monitoring, ensuring your organization is audit-ready at all times.

Our platform specifically addresses three critical pain points:
*   **Automated Gap Analysis:** Our AI engine maps your existing infrastructure against SOC2 controls to identify deficiencies in real-time.
*   **Continuous Evidence Collection:** We integrate directly with your tech stack to automatically gather and verify audit evidence, eliminating the "crunch" before an audit.
*   **AI-Generated Policy Frameworks

In [39]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    params: resend.Emails.SendParams = {
    "from": "Acme <onboarding@resend.dev>",
    "to": ["jayanthreddykonda1@gmail.com"],
    "subject": "Sales",
    "html": f"{body}",
    }
    resend.Emails.send(params)
    return {"status": "success"}

In [40]:
description = "Write a cold sales email"

Agent1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
Agent2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
Agent3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [41]:
tools = [Agent1, Agent2, Agent3, send_email]

In [43]:
instructions = """
You are a Sales Manager at CoolAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=gemini_model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)